In [2]:
from pathlib import Path
import pandas as pd
import numpy as np


In [1]:
def get_participant_files(data_dir, start=1, end=60):

    files = sorted(
        data_dir.rglob("full_gsr_ppg*.csv"),
        key=lambda x: int(x.parent.name.replace("Part", ""))
    )

    files = [
        f for f in files
        if start <= int(f.parent.name.replace("Part", "")) <= end
    ]

    return files

In [3]:

DATA_DIR = Path("../data/raw/Participants")

# -------------------------
# LOAD FILES
# -------------------------
participant_files = get_participant_files(DATA_DIR, 1, 48)

results = []

for file in participant_files:

    participant_id = file.parent.name
    print(f"\nProcessing: {participant_id}")

    df = pd.read_csv(file)

    # -------------------------
    # EXTRACT SIGNALS
    # -------------------------
    gsr = pd.to_numeric(df["gsr"], errors="coerce")
    ppg = pd.to_numeric(df["ppg"], errors="coerce")

    # -------------------------
    # BASIC STATS
    # -------------------------
    gsr_min, gsr_max = np.nanmin(gsr), np.nanmax(gsr)
    gsr_std = np.nanstd(gsr)
    gsr_nan = gsr.isna().mean()

    ppg_min, ppg_max = np.nanmin(ppg), np.nanmax(ppg)
    ppg_std = np.nanstd(ppg)
    ppg_nan = ppg.isna().mean()

    duration_sec = len(df) / 256  # raw sampling rate

    # -------------------------
    # QUALITY FLAGS
    # -------------------------
    gsr_flag = "OK"
    if gsr_nan > 0.2:
        gsr_flag = "NaN"
    elif gsr_std < 1e-3:
        gsr_flag = "Flat"
    elif gsr_max > 10000:
        gsr_flag = "Extreme"

    ppg_flag = "OK"
    if ppg_nan > 0.2:
        ppg_flag = "NaN"
    elif ppg_std < 1e-3:
        ppg_flag = "Flat"

    print(f"GSR → min:{gsr_min:.2f} max:{gsr_max:.2f} std:{gsr_std:.2f} NaN:{gsr_nan:.2f} [{gsr_flag}]")
    print(f"PPG → min:{ppg_min:.2f} max:{ppg_max:.2f} std:{ppg_std:.2f} NaN:{ppg_nan:.2f} [{ppg_flag}]")

    results.append({
        "participant": participant_id,
        "gsr_min": gsr_min,
        "gsr_max": gsr_max,
        "gsr_std": gsr_std,
        "gsr_nan": gsr_nan,
        "gsr_flag": gsr_flag,
        "ppg_min": ppg_min,
        "ppg_max": ppg_max,
        "ppg_std": ppg_std,
        "ppg_nan": ppg_nan,
        "ppg_flag": ppg_flag,
        "duration_sec": duration_sec
    })

# -------------------------
# RESULT TABLE
# -------------------------
results_df = pd.DataFrame(results)

print("\n=== SUMMARY ===")
print(results_df["gsr_flag"].value_counts())
print(results_df["ppg_flag"].value_counts())

# -------------------------
# SAVE
# -------------------------
results_df.to_csv("../data/processed/signal_quality_report.csv", index=False)

results_df.head()


Processing: Part1
GSR → min:52.92 max:36621.95 std:376.90 NaN:0.00 [Extreme]
PPG → min:0.00 max:1846.89 std:71.57 NaN:0.00 [OK]

Processing: Part2
GSR → min:76.53 max:220.46 std:24.77 NaN:0.00 [OK]
PPG → min:0.00 max:2385.35 std:185.78 NaN:0.00 [OK]

Processing: Part3
GSR → min:131.68 max:280.35 std:32.42 NaN:0.00 [OK]
PPG → min:0.00 max:2775.09 std:61.39 NaN:0.00 [OK]

Processing: Part4
GSR → min:22.70 max:54.06 std:4.68 NaN:0.00 [OK]
PPG → min:0.00 max:1730.40 std:71.81 NaN:0.00 [OK]

Processing: Part5
GSR → min:37.30 max:145.58 std:21.46 NaN:0.00 [OK]
PPG → min:0.00 max:1630.04 std:79.47 NaN:0.00 [OK]

Processing: Part6
GSR → min:71.73 max:463.50 std:61.41 NaN:0.00 [OK]
PPG → min:0.00 max:1989.01 std:104.88 NaN:0.00 [OK]

Processing: Part7
GSR → min:28.39 max:99.51 std:10.35 NaN:0.00 [OK]
PPG → min:0.00 max:2172.89 std:125.21 NaN:0.00 [OK]

Processing: Part8
GSR → min:92.53 max:243.71 std:17.30 NaN:0.00 [OK]
PPG → min:0.00 max:2794.14 std:113.82 NaN:0.00 [OK]

Processing: Part9
GSR

,participant,gsr_min,gsr_max,gsr_std,gsr_nan,gsr_flag,ppg_min,ppg_max,ppg_std,ppg_nan,ppg_flag,duration_sec
0,Part1,52.915140,36621.951220,376.900016,0.000637,Extreme,0.0,1846.886447,71.572713,0.000637,OK,2178.332031
1,Part2,76.531381,220.458638,24.773307,0.000588,OK,0.0,2385.347985,185.780833,0.000588,OK,2339.894531
2,Part3,131.682353,280.345040,32.415885,0.000571,OK,0.0,2775.091575,61.394057,0.000571,OK,2339.656250
3,Part4,22.702938,54.062069,4.677775,0.000952,OK,0.0,1730.402930,71.812266,0.000952,OK,1477.679688
4,Part5,37.303195,145.579710,21.463866,0.000632,OK,0.0,1630.036630,79.474054,0.000632,OK,2150.816406
